# Solutions · 11 · Kalman filtering and LQG

Worked solutions to the exercises of [notebook 11](../notebooks/11_kalman_filtering_and_lqg.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, statespace as sp, discrete as dc, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Run the Kalman filter with a measurement noise variance ten times smaller than the true one (an over-confident filter) and ten times larger; compare the estimation error variances with the Riccati predictions. Which direction of mis-tuning is worse?

In [2]:
mot = models.dc_motor()["system"]
dt = 0.005
Gd = dc.c2d(mot, dt); Ad, Bd, Cd = Gd.A, Gd.B, Gd.C
Qd, Rd_true = np.diag([1e-6, 1e-4]), 0.25
rng = np.random.default_rng(3)
N = 6000
u = 6.0 * (1 + np.sign(np.sin(0.5 * np.arange(N) * dt)))
x = np.zeros(2); X = np.zeros((N, 2)); Y = np.zeros(N)
for k in range(N):
    X[k] = x; Y[k] = (Cd @ x)[0] + np.sqrt(Rd_true) * rng.normal()
    x = Ad @ x + Bd[:, 0] * u[k] + rng.normal(size=2) * np.sqrt(np.diag(Qd))
for factor, label in ((0.1, "over-confident (R/10)"), (1.0, "correct"), (10.0, "over-cautious (10 R)")):
    kd = sp.kalman(Ad, Cd, Qd, np.array([[Rd_true * factor]]), discrete=True)
    Xh = sp.kalman_filter_run(Ad, Bd, Cd, kd["L"], u, Y)
    e = Xh[500:] - X[500:]
    print(f"{label:<22}: speed error variance {np.var(e[:, 1]):.5f} (filter predicts {kd['P'][1, 1]:.5f}), current error variance {np.var(e[:, 0]):.2e} (predicts {kd['P'][0, 0]:.2e})")

over-confident (R/10) : speed error variance 0.00187 (filter predicts 0.00081), current error variance 4.65e-05 (predicts 5.05e-05)
correct               : speed error variance 0.00088 (filter predicts 0.00101), current error variance 4.65e-05 (predicts 5.05e-05)
over-cautious (10 R)  : speed error variance 0.00092 (filter predicts 0.00105), current error variance 4.65e-05 (predicts 5.05e-05)


An over-confident filter (told the measurement is cleaner than it is) weights it too much and lets noise
into the estimate: its actual error exceeds its own prediction several times over. An over-cautious one
trusts the model and lags the truth less visibly here, because this model is good. Measured error above the
predicted one is the signature of a filter that believes its sensor too much; it is the worse direction,
because the filter then also reports a confidence it does not have.

## Exercise 2

Loop transfer recovery: for the DC motor LQG, scale the process noise covariance by 1, 10^2, 10^4, 10^6 and tabulate the margins at the plant input and the rms of the control signal in response to measurement noise. Where does the recovery stop paying?

In [3]:
from englincontrol import robust as rb, response as rs
cp = models.cart_pendulum()["system"]
Q, R = np.diag([10.0, 1.0, 100.0, 1.0]), np.array([[0.1]])
rng = np.random.default_rng(0)
K_lqr = sp.lqr(cp.A, cp.B, Q, R)["K"]
print(f"state feedback: closest approach to -1: {rb.nyquist_distance(sy.ss(cp.A, cp.B, K_lqr, np.zeros((1, 1))).to_tf())['distance']:.3f}")
rows = []
for scale in (1e-6, 1e-3, 1.0, 1e2):
    g = sp.lqg(cp, Q, R, scale * np.eye(4), 1e-4 * np.eye(2))
    comp = g["compensator"]
    L_in = sy.series(cp, sy.ss(comp.A, comp.B, -comp.C, np.zeros((1, 2)))).to_tf()
    d = rb.nyquist_distance(L_in)
    t = np.linspace(0, 1, 2001)
    _, u_noise, _ = rs.lsim(comp, 1e-3 * rng.normal(size=(2001, 2)), t)          # 1 mrad / 1 mm of measurement noise on both outputs
    rows.append((scale, f"{np.min(g['estimator_poles'].real):.0f} .. {np.max(g['estimator_poles'].real):.1f}", d["distance"], np.std(u_noise)))
print(pd.DataFrame(rows, columns=["process noise scale", "estimator poles (Re)", "closest approach to -1", "control noise rms (N)"]).to_string(index=False, float_format="%.3f"))

state feedback: closest approach to -1: 1.000
 process noise scale estimator poles (Re)  closest approach to -1  control noise rms (N)
               0.000           -6 .. -0.2                   0.215                  0.003
               0.001           -7 .. -1.1                   0.286                  0.003
               1.000         -100 .. -1.1                   0.461                  0.011
             100.000        -1000 .. -1.0                   0.500                  0.036


Scaling up the assumed process noise makes the Kalman filter trust the measurement more and its poles faster;
the loop's distance from −1 recovers from 0.21 towards the state-feedback value of 1 - loop transfer
recovery - but saturates at 0.5 while the control signal's response to measurement noise keeps growing, by
twentyfold across the sweep. The recovery stops paying at a process-noise scale of about 1 here; and it can
never complete, because the cart position is a non-minimum-phase output of this plant, for which the
recovery theorem does not hold.

## Exercise 3

**Implement it yourself:** write the time-varying discrete Kalman filter (the Riccati recursion P_{k+1} = A P A' - A P C'(C P C' + R)^-1 C P A' + Q together with the gain) and show that its gain converges to the steady-state one of `statespace.kalman`.

In [4]:
mot = models.dc_motor()["system"]
dt = 0.005
Gd = dc.c2d(mot, dt); Ad, Cd = Gd.A, Gd.C
Qd, Rd = np.diag([1e-6, 1e-4]), np.array([[0.25]])
P = np.zeros((2, 2)); gains = []
for k in range(400):
    L = Ad @ P @ Cd.T @ np.linalg.inv(Cd @ P @ Cd.T + Rd)
    P = Ad @ P @ Ad.T - Ad @ P @ Cd.T @ np.linalg.inv(Cd @ P @ Cd.T + Rd) @ Cd @ P @ Ad.T + Qd
    gains.append(L.ravel())
gains = np.array(gains)
Lss = sp.kalman(Ad, Cd, Qd, Rd, discrete=True)["L"].ravel()
print("gain after 50, 200, 400 steps:", np.round(gains[49], 5), np.round(gains[199], 5), np.round(gains[399], 5), " steady state:", np.round(Lss, 5))
print(f"difference at step 400: {np.max(np.abs(gains[399] - Lss)):.1e}")

gain after 50, 200, 400 steps: [0.      0.00382] [1.00e-05 3.84e-03] [1.00e-05 3.84e-03]  steady state: [1.00e-05 3.84e-03]
difference at step 400: 7.7e-09


The time-varying filter's Riccati recursion starts from any P₀ and converges to the steady-state solution of
the discrete algebraic Riccati equation; its gain converges with it, within a few hundred samples here. The
steady-state filter is this limit, and is what one implements when the transient of the first seconds does not
matter.